# VLADE · QwenVLA 백본 학습 범위 비교

Colab Secrets에 `HF_TOKEN`을 설정한 다음 `plan → setup → preflight → pilot → train` 순서로 실행합니다.
설정을 바꾸면 새 `RUN_GROUP`을 사용합니다. 재개할 때는 원래 `SOURCE_COMMIT`과 `RUN_GROUP`을 지정합니다.
자세한 설정과 로그 위치는 [실행 안내](../docs/environment/action_level_distillation.md)를 참고하십시오.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys, time
os.environ.setdefault('VLADE_SESSION_STARTED_UNIX', str(time.time()))
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
SOURCE_REF = 'refs/heads/main'  # 재개는 최초 실행의 SOURCE_COMMIT을 지정합니다.
REPOSITORY = 'https://github.com/eastha10/VLADE.git'
SOURCE_COMMIT = SOURCE_REF if re.fullmatch(r'[0-9a-f]{40}', SOURCE_REF) else subprocess.check_output(['git','ls-remote',REPOSITORY,SOURCE_REF], text=True).split()[0]
if not re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT):
    raise ValueError('불변 Git commit이 필요합니다.')
CODE = Path('/content/vlade_sources') / SOURCE_COMMIT
if any((name == 'vlade' or name.startswith('vlade.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(CODE) for name,module in list(sys.modules.items())):
    raise RuntimeError('다른 소스가 로드되어 있습니다. Colab 런타임을 다시 시작하십시오.')
if not CODE.exists():
    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',REPOSITORY,str(CODE)],check=True)
    subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
    subprocess.run(['git','-C',str(CODE),'checkout','--detach',SOURCE_COMMIT],check=True)
sys.path[:0] = [str(CODE/'src'),str(CODE)]
from vlade.utils.source_identity import source_identity
assert source_identity(CODE,require_clean=True)['git_commit']==SOURCE_COMMIT
print('SOURCE_COMMIT:',SOURCE_COMMIT)


In [ ]:
from vlade.research.distillation import new_run_group

CONFIG = 'configs/experiments/qwen_vla_action_kd.json'
WORKSPACE = '/content/vlade_distillation'
RUN_GROUP = globals().get('RUN_GROUP') or new_run_group('qwen-vla-scope')
ACTION = "plan"  #@param ["plan", "setup", "preflight", "pilot", "train", "retry"]
OVERRIDES = {'workflow.variant': 'gt_only'}
RESUME_REVISIONS = {}  # 예: {'gt_kd': '40자리 HF commit SHA'}


In [ ]:
from vlade.research.distillation import job

job(CODE / CONFIG, workspace=WORKSPACE, run_group=RUN_GROUP, action=ACTION,
    overrides=OVERRIDES, resume_revisions=RESUME_REVISIONS, colab=True)
